# AI Agents: Introduction and Implementation

## What is an AI Agent?

An **AI Agent** is an autonomous system that can:
- **Perceive** its environment through inputs
- **Reason** about what actions to take
- **Act** by executing tasks and using tools
- **Learn** from feedback to improve performance

### Key Components of AI Agents:
1. **LLM Brain**: Large Language Model for reasoning and decision-making
2. **Tools**: Functions the agent can call (APIs, databases, calculators, etc.)
3. **Memory**: Short-term (conversation) and long-term (persistent) storage
4. **Planning**: Breaking down complex tasks into steps
5. **Reflection**: Self-evaluation and error correction

### Types of AI Agents:
- **ReAct Agents**: Reasoning + Acting in iterative loops
- **Planning Agents**: Plan first, then execute
- **Autonomous Agents**: Continuous goal-seeking behavior
- **Multi-Agent Systems**: Multiple specialized agents collaborating

## Setup and Installation

In [1]:
# Install required packages
!pip install openai python-dotenv requests wikipedia-api -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 1.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
import os
import json
import requests
from datetime import datetime
from typing import List, Dict, Callable
import openai
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
openai.api_key = os.getenv('OPENAI_API_KEY')

#or set it in the python code os.environ["OPENAI_API_KEY"] =  your key



## Building a Simple ReAct Agent

We'll implement a **ReAct (Reasoning + Acting)** agent that can:
- Use multiple tools
- Reason about which tool to use
- Execute actions
- Provide answers based on tool outputs

### Step 1: Define Tools

Tools are functions that the agent can call to interact with the world.

In [3]:
# Tool 1: Calculator
def calculator(expression: str) -> str:
    """Evaluates a mathematical expression."""
    try:
        result = eval(expression)
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {str(e)}"

# Tool 2: Wikipedia Search
def wikipedia_search(query: str) -> str:
    """Searches Wikipedia for information about a topic."""
    try:
        import wikipediaapi
        wiki = wikipediaapi.Wikipedia(
            user_agent='StudentLearningAgent/1.0 (Educational Purpose)',
            language='en'
        )
        page = wiki.page(query)
        if page.exists():
            # Return first 500 characters
            return page.summary[:500] + "..."
        else:
            return f"No Wikipedia page found for '{query}'"
    except Exception as e:
        return f"Error searching Wikipedia: {str(e)}"

# Tool 3: Current Date/Time
def get_current_datetime() -> str:
    """Returns the current date and time."""
    now = datetime.now()
    return now.strftime("%Y-%m-%d %H:%M:%S")

# Tool 4: Weather Information (Mock)
def get_weather(city: str) -> str:
    """Gets weather information for a city (mock data)."""
    # In a real implementation, you'd call a weather API
    weather_data = {
        "london": "Cloudy, 15°C",
        "new york": "Sunny, 22°C",
        "tokyo": "Rainy, 18°C",
        "paris": "Partly cloudy, 17°C"
    }
    city_lower = city.lower()
    return weather_data.get(city_lower, f"Weather data not available for {city}")

print("✓ Tools defined successfully")

✓ Tools defined successfully


### Step 2: Create Tool Registry

We need to describe our tools in a format the LLM can understand.

In [4]:
# Tool descriptions for the LLM
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluates mathematical expressions. Use for calculations.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The mathematical expression to evaluate, e.g., '2+2' or '10*5'"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "wikipedia_search",
            "description": "Searches Wikipedia for information about a topic. Use when you need factual information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The topic to search for on Wikipedia"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_datetime",
            "description": "Returns the current date and time. Use when asked about the current date or time.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Gets current weather information for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

# Map function names to actual functions
AVAILABLE_FUNCTIONS = {
    "calculator": calculator,
    "wikipedia_search": wikipedia_search,
    "get_current_datetime": get_current_datetime,
    "get_weather": get_weather
}

print(f"✓ Registered {len(TOOLS)} tools")

✓ Registered 4 tools


### Step 3: Implement the Agent

The agent follows a ReAct loop:
1. **Think**: Reason about what to do
2. **Act**: Call a tool
3. **Observe**: Get the result
4. **Repeat** until the task is complete

In [5]:
class SimpleAgent:
    """A simple ReAct agent that can use tools to answer questions."""

    def __init__(self, model="gpt-4o-mini", max_iterations=5):
        self.model = model
        self.max_iterations = max_iterations
        self.conversation_history = []

    def run(self, user_query: str, verbose=True) -> str:
        """Run the agent on a user query."""
        # Initialize conversation
        self.conversation_history = [
            {
                "role": "system",
                "content": """You are a helpful AI agent with access to tools.
                When asked a question:
                1. Think about what information you need
                2. Use the appropriate tool(s) to gather information
                3. Synthesize the information to answer the question

                Be concise and accurate. Always use tools when you need current information."""
            },
            {
                "role": "user",
                "content": user_query
            }
        ]

        if verbose:
            print(f"\n{'='*60}")
            print(f"🤖 Agent started")
            print(f"📝 Query: {user_query}")
            print(f"{'='*60}\n")

        # ReAct loop
        for iteration in range(self.max_iterations):
            if verbose:
                print(f"\n--- Iteration {iteration + 1} ---")

            # Call LLM
            response = openai.chat.completions.create(
                model=self.model,
                messages=self.conversation_history,
                tools=TOOLS,
                tool_choice="auto"
            )

            assistant_message = response.choices[0].message
            self.conversation_history.append(assistant_message)

            # Check if the agent wants to use a tool
            if assistant_message.tool_calls:
                # Execute each tool call
                for tool_call in assistant_message.tool_calls:
                    function_name = tool_call.function.name
                    function_args = json.loads(tool_call.function.arguments)

                    if verbose:
                        print(f"\n🔧 Calling tool: {function_name}")
                        print(f"   Arguments: {function_args}")

                    # Execute the function
                    function_to_call = AVAILABLE_FUNCTIONS[function_name]
                    function_response = function_to_call(**function_args)

                    if verbose:
                        print(f"   Result: {function_response}")

                    # Add tool response to conversation
                    self.conversation_history.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": function_name,
                        "content": function_response
                    })
            else:
                # No more tool calls, return final answer
                final_answer = assistant_message.content
                if verbose:
                    print(f"\n{'='*60}")
                    print(f"✅ Final Answer:\n{final_answer}")
                    print(f"{'='*60}\n")
                return final_answer

        return "Max iterations reached without final answer."

print("✓ Agent class created")

✓ Agent class created


## Testing the Agent

Let's test our agent with various queries that require different tools.

### Test 1: Mathematical Calculation

In [17]:
agent = SimpleAgent()
answer = agent.run("What is 234 * 567?")


🤖 Agent started
📝 Query: What is 234 * 567?


--- Iteration 1 ---

🔧 Calling tool: calculator
   Arguments: {'expression': '234*567'}
   Result: Result: 132678

--- Iteration 2 ---

✅ Final Answer:
The result of \( 234 \times 567 \) is 132,678.



### Test 2: Information Retrieval

In [18]:
answer = agent.run("Tell me about the Eiffel Tower")


🤖 Agent started
📝 Query: Tell me about the Eiffel Tower


--- Iteration 1 ---

🔧 Calling tool: wikipedia_search
   Arguments: {'query': 'Eiffel Tower'}
   Result: Error searching Wikipedia: Please, be nice to Wikipedia and specify user agent - https://meta.wikimedia.org/wiki/User-Agent_policy. Current user_agent: 'en' is not sufficient. Use Wikipedia(user_agent='your-user-agent', language='en')

--- Iteration 2 ---

🔧 Calling tool: wikipedia_search
   Arguments: {'query': 'Eiffel Tower'}
   Result: Error searching Wikipedia: Please, be nice to Wikipedia and specify user agent - https://meta.wikimedia.org/wiki/User-Agent_policy. Current user_agent: 'en' is not sufficient. Use Wikipedia(user_agent='your-user-agent', language='en')

--- Iteration 3 ---

✅ Final Answer:
The Eiffel Tower is a wrought-iron tower located in Paris, France. It was designed by the engineer Gustave Eiffel and completed in 1889 as the entrance arch to the 1889 World's Fair (Exposition Universelle) held to celebra

### Test 3: Current Information

In [19]:
answer = agent.run("What's the date today?")


🤖 Agent started
📝 Query: What's the date today?


--- Iteration 1 ---

🔧 Calling tool: get_current_datetime
   Arguments: {}
   Result: 2026-03-06 06:49:54

--- Iteration 2 ---

✅ Final Answer:
Today's date is March 6, 2026.



### Test 4: Multi-Step Reasoning

In [20]:
answer = agent.run("What's the weather in Paris, and tell me one famous landmark there?")


🤖 Agent started
📝 Query: What's the weather in Paris, and tell me one famous landmark there?


--- Iteration 1 ---

🔧 Calling tool: get_weather
   Arguments: {'city': 'Paris'}
   Result: Partly cloudy, 17°C

🔧 Calling tool: wikipedia_search
   Arguments: {'query': 'landmarks in Paris'}
   Result: Error searching Wikipedia: Please, be nice to Wikipedia and specify user agent - https://meta.wikimedia.org/wiki/User-Agent_policy. Current user_agent: 'en' is not sufficient. Use Wikipedia(user_agent='your-user-agent', language='en')

--- Iteration 2 ---

🔧 Calling tool: get_weather
   Arguments: {'city': 'Paris'}
   Result: Partly cloudy, 17°C

🔧 Calling tool: wikipedia_search
   Arguments: {'query': 'Eiffel Tower'}
   Result: Error searching Wikipedia: Please, be nice to Wikipedia and specify user agent - https://meta.wikimedia.org/wiki/User-Agent_policy. Current user_agent: 'en' is not sufficient. Use Wikipedia(user_agent='your-user-agent', language='en')

--- Iteration 3 ---

🔧 Calling t

### Test 5: Complex Query Requiring Multiple Tools

In [21]:
answer = agent.run(
    "If I have 150 euros and the Eiffel Tower ticket costs 28.30 euros, "
    "how many tickets can I buy? Also, what's the weather like in Paris?"
)


🤖 Agent started
📝 Query: If I have 150 euros and the Eiffel Tower ticket costs 28.30 euros, how many tickets can I buy? Also, what's the weather like in Paris?


--- Iteration 1 ---

🔧 Calling tool: calculator
   Arguments: {'expression': '150/28.30'}
   Result: Result: 5.300353356890459

🔧 Calling tool: get_weather
   Arguments: {'city': 'Paris'}
   Result: Partly cloudy, 17°C

--- Iteration 2 ---

✅ Final Answer:
You can buy 5 tickets to the Eiffel Tower with your 150 euros, as you can only purchase whole tickets. As for the weather in Paris, it is partly cloudy with a temperature of 17°C.



## Advanced Features

### Adding Memory to the Agent

In [22]:
class AgentWithMemory(SimpleAgent):
    """Agent with conversation memory across multiple queries."""

    def __init__(self, model="gpt-4o-mini", max_iterations=5):
        super().__init__(model, max_iterations)
        # Persistent conversation history
        self.conversation_history = [
            {
                "role": "system",
                "content": """You are a helpful AI agent with access to tools and memory.
                You can remember previous interactions in this conversation.
                When asked a question, consider the context of previous queries."""
            }
        ]

    def run(self, user_query: str, verbose=True) -> str:
        """Run the agent with memory of previous interactions."""
        # Add new user query
        self.conversation_history.append({
            "role": "user",
            "content": user_query
        })

        if verbose:
            print(f"\n{'='*60}")
            print(f"🤖 Agent with Memory")
            print(f"📝 Query: {user_query}")
            print(f"{'='*60}\n")

        # ReAct loop (similar to before)
        for iteration in range(self.max_iterations):
            if verbose:
                print(f"\n--- Iteration {iteration + 1} ---")

            response = openai.chat.completions.create(
                model=self.model,
                messages=self.conversation_history,
                tools=TOOLS,
                tool_choice="auto"
            )

            assistant_message = response.choices[0].message
            self.conversation_history.append(assistant_message)

            if assistant_message.tool_calls:
                for tool_call in assistant_message.tool_calls:
                    function_name = tool_call.function.name
                    function_args = json.loads(tool_call.function.arguments)

                    if verbose:
                        print(f"\n🔧 Tool: {function_name}({function_args})")

                    function_to_call = AVAILABLE_FUNCTIONS[function_name]
                    function_response = function_to_call(**function_args)

                    if verbose:
                        print(f"   Result: {function_response}")

                    self.conversation_history.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": function_name,
                        "content": function_response
                    })
            else:
                final_answer = assistant_message.content
                if verbose:
                    print(f"\n{'='*60}")
                    print(f"✅ Answer: {final_answer}")
                    print(f"{'='*60}\n")
                return final_answer

        return "Max iterations reached."

print("✓ Agent with memory created")

✓ Agent with memory created


### Testing Agent with Memory

In [23]:
# Create agent with memory
memory_agent = AgentWithMemory()

# First query
memory_agent.run("What's 25 * 4?")


🤖 Agent with Memory
📝 Query: What's 25 * 4?


--- Iteration 1 ---

🔧 Tool: calculator({'expression': '25*4'})
   Result: Result: 100

--- Iteration 2 ---

✅ Answer: 25 * 4 equals 100.



'25 * 4 equals 100.'

In [24]:
# Second query - agent remembers the previous calculation
memory_agent.run("Now add 50 to that result")


🤖 Agent with Memory
📝 Query: Now add 50 to that result


--- Iteration 1 ---

🔧 Tool: calculator({'expression': '100+50'})
   Result: Result: 150

--- Iteration 2 ---

✅ Answer: 100 + 50 equals 150.



'100 + 50 equals 150.'

In [14]:
# Third query - testing contextual memory
memory_agent.run("What was my first question?")


🤖 Agent with Memory
📝 Query: What was my first question?


--- Iteration 1 ---

✅ Answer: Your first question was, "What's 25 * 4?"



'Your first question was, "What\'s 25 * 4?"'

## Agent Design Patterns

### 1. ReAct Pattern (What we implemented)
- **Reason** → **Act** → **Observe** loop
- Best for: General-purpose tasks, exploratory problems

### 2. Plan-and-Execute Pattern
- Create a plan first
- Execute steps sequentially
- Best for: Complex multi-step tasks

### 3. Reflexion Pattern
- Execute → Evaluate → Refine
- Agent reflects on its mistakes
- Best for: Tasks requiring high accuracy

### 4. Multi-Agent Pattern
- Multiple specialized agents
- Coordinate and communicate
- Best for: Complex systems, different expertise domains

## Adding Custom Tools

Example: Adding a file reading tool

In [15]:
def read_file(filename: str) -> str:
    """Reads content from a file."""
    try:
        with open(filename, 'r') as f:
            content = f.read()
        return content[:1000]  # Return first 1000 chars
    except Exception as e:
        return f"Error reading file: {str(e)}"

# Add to tools registry
new_tool = {
    "type": "function",
    "function": {
        "name": "read_file",
        "description": "Reads the contents of a file",
        "parameters": {
            "type": "object",
            "properties": {
                "filename": {
                    "type": "string",
                    "description": "Path to the file to read"
                }
            },
            "required": ["filename"]
        }
    }
}

print("✓ Custom tool 'read_file' ready to be added")

✓ Custom tool 'read_file' ready to be added


## Best Practices for AI Agents

### 1. Tool Design
- ✅ Keep tools focused and single-purpose
- ✅ Provide clear descriptions
- ✅ Handle errors gracefully
- ✅ Return structured, parseable outputs

### 2. Prompt Engineering
- ✅ Be explicit about the agent's role
- ✅ Provide examples of tool usage
- ✅ Set clear success criteria
- ✅ Include safety guidelines

### 3. Error Handling
- ✅ Set maximum iteration limits
- ✅ Implement fallback strategies
- ✅ Log agent actions for debugging
- ✅ Validate tool inputs and outputs

### 4. Performance
- ✅ Use faster models (gpt-4o-mini) when possible
- ✅ Cache frequent queries
- ✅ Limit tool call depth
- ✅ Implement streaming for long responses